# Human in the loop – zadania


Zaimplementuj interaktywnego chatbota wymieniającego po jednej wiadomości z użytkownikiem. Historia rozmowy powinna być zapamiętywana między turami dzięki reduktorowi `add_messages` i checkpointerowi.

## Stan (TypedDict)

1. Zdefiniuj `State` zawierający:
- `messages: Annotated[list, add_messages]` – historia wiadomości z reduktorem dopisującym nowe wiadomości zamiast nadpisywać listę
- `turn_count: int` – liczba ukończonych rund rozmowy
- `active: bool` – czy rozmowa jest aktywna

Wskazówka: `add_messages` importujesz z `langgraph.graph.message`.

---
(czas: 6 min.)

In [ ]:
# ...

## Node'y

2. Napisz node `chat_node(state: State, config: RunnableConfig)`, który:
- pobiera `model_name` i `system_prompt` z `configurable`
- inicjalizuje `ChatOpenAI`
- wywołuje model na historii wiadomości (dodaj `SystemMessage` z system_promptem na początku listy)
- zwraca odpowiedź jako `AIMessage` w polu `messages`
- zwiększa `turn_count` o 1

---
(czas: 12 min.)

In [ ]:
# ...

3. Napisz node `human_input_node(state: State)`, który:
- zatrzymuje wykonanie grafu przez `interrupt("Ty: ")` i czeka na wiadomość użytkownika
- zapisuje otrzymaną wartość jako `HumanMessage` w `messages`
- ustawia `active = False` jeśli użytkownik wpisał jedno ze słów kończących: `koniec`, `exit`, `quit`, `bye`, `stop`

Wskazówka: `interrupt` i `Command` importujesz z `langgraph.types`.

---
(czas: 10 min.)

In [ ]:
# ...

## Graf i checkpointer

4. Napisz funkcję routingu `should_continue(state: State) -> str`, która zwraca `"human_input"` gdy `active == True` i `"end"` gdy `active == False`.

Następnie zbuduj `StateGraph` z następującą strukturą:
- `START → human_input`
- `human_input → chat`
- `chat → conditional (should_continue) → human_input` lub `END`

Skompiluj pipeline z `checkpointer=MemorySaver()` (wymagane dla `interrupt`). Wyświetl wizualizację.

---
(czas: 12 min.)

In [ ]:
# ...

## Uruchomienie chatbota

5. Uruchom chatbota w pętli. Schemat działania:

1. Zdefiniuj `config = {"configurable": {"thread_id": "1", "model_name": ..., "system_prompt": ...}}`
2. Uruchom pipeline: `pipeline.invoke({"messages": [], "turn_count": 0, "active": True}, config=config)` – pipeline zatrzyma się na pierwszym `interrupt`
3. W pętli `while True`:
   - Sprawdź `pipeline.get_state(config).next` – jeśli puste, zakończ
   - Pobierz wiadomość od użytkownika (`input()`)
   - Wznów pipeline: `pipeline.invoke(Command(resume=user_input), config=config)`

Wskazówka: Wypisuj odpowiedź chatbota po każdym `invoke` – znajdziesz ją w ostatnim `AIMessage` w historii wiadomości.

Jeśli zdążysz:
- Wyświetlaj pełną historię rozmowy po każdej turze
- Dodaj limit `max_turns` – zakończ rozmowę gdy liczba tur osiągnie limit (zmień funkcję routingu)

---
(czas: 15 min.)

In [ ]:
# ...